# 筛选变量

## 计算相关性

In [24]:
import os
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# 路径
DATA_DIR = r"E:\Project-yqr\828update\BD\results"
YEARS = [2009, 2015, 2018]

def p_label(p):
    if p < 0.01:
        return "<0.01"
    elif p < 0.05:
        return "<0.05"
    else:
        return ">0.05"

# 存储结果
results = {}
all_data = []

for year in YEARS:
    path = os.path.join(DATA_DIR, f"soc_filled_{year}_average.csv")
    df = pd.read_csv(path)
    
    # 强制转为数值型
    soc = df["SOC"].astype(float)
    sub_df = df.iloc[:, 7:]
    sub_df = sub_df.astype(float)

    corr_dict = {}
    for col in sub_df.columns:
        x, y = soc, sub_df[col]
        mask = (~x.isna()) & (~y.isna()) & (~x.isin([np.inf, -np.inf])) & (~y.isin([np.inf, -np.inf]))
        if mask.sum() > 2:
            r, p = pearsonr(x[mask], y[mask])
            corr_dict[col] = (r, p)
        else:
            corr_dict[col] = (np.nan, np.nan)
    results[year] = corr_dict
    all_data.append(df)

    # 检查输出
    print(f"✅ {year} 协变量数量: {sub_df.shape[1]}, 相关性结果数量: {len(corr_dict)}")

# All 数据
df_all = pd.concat(all_data, axis=0, ignore_index=True)
soc_all = df_all["SOC"].astype(float)
sub_df_all = df_all.iloc[:, 7:].astype(float)

all_corr = {}
for col in sub_df_all.columns:
    x, y = soc_all, sub_df_all[col]
    mask = (~x.isna()) & (~y.isna()) & (~x.isin([np.inf, -np.inf])) & (~y.isin([np.inf, -np.inf]))
    if mask.sum() > 2:
        r, p = pearsonr(x[mask], y[mask])
        all_corr[col] = (r, p)
    else:
        all_corr[col] = (np.nan, np.nan)
results["All"] = all_corr

# 检查 All
print(f"✅ All 协变量数量: {sub_df_all.shape[1]}, 相关性结果数量: {len(all_corr)}")

# 按每年排序（基于 |Corr|）
sorted_dfs = {}
for year in YEARS + ["All"]:
    year_df = pd.DataFrame([
        {"Variable": col, "Corr": r, "AbsCorr": abs(r), "p": p_label(p)}
        for col, (r, p) in results[year].items()
    ])
    # 按绝对值排序，但保留原始Corr
    year_df = year_df.sort_values("AbsCorr", ascending=False).reset_index(drop=True)
    year_df = year_df.drop(columns=["AbsCorr"])  # 输出不需要显示AbsCorr
    sorted_dfs[year] = year_df

# 对齐并列展示
max_len = max(len(df) for df in sorted_dfs.values())
for year in sorted_dfs:
    sorted_dfs[year] = sorted_dfs[year].reindex(range(max_len))

# 拼接成一个表
df_combined = pd.concat(
    [sorted_dfs[year].rename(columns={
        "Variable": f"{year}_Variable",
        "Corr": f"{year}_Corr",
        "p": f"{year}_p"
    }) for year in YEARS + ["All"]],
    axis=1
)

out_path = os.path.join(DATA_DIR, "SOC_correlations_sorted_abs_side_by_side.csv")
df_combined.to_csv(out_path, index=False)

print(f"🎯 已保存按绝对值排序的结果: {out_path}")


✅ 2009 协变量数量: 107, 相关性结果数量: 107
✅ 2015 协变量数量: 107, 相关性结果数量: 107
✅ 2018 协变量数量: 107, 相关性结果数量: 107
✅ All 协变量数量: 107, 相关性结果数量: 107
🎯 已保存按绝对值排序的结果: E:\Project-yqr\828update\BD\results\SOC_correlations_sorted_abs_side_by_side.csv


## 计算自相关

In [26]:
import os
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# ============================
# 路径 & 参数
# ============================
DATA_DIR = r"E:\Project-yqr\828update\BD\results"
files = {
    "2009": os.path.join(DATA_DIR, "soc_filled_2009_average.csv"),
    "2015": os.path.join(DATA_DIR, "soc_filled_2015_average.csv"),
    "2018": os.path.join(DATA_DIR, "soc_filled_2018_average.csv"),
}
threshold = 0.75  # 高相关性阈值

def significance_marker(p):
    if p < 0.01:
        return "<0.01"
    elif p < 0.05:
        return "<0.05"
    else:
        return "≥0.05"

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

# ============================
# 1. 读取数据
# ============================
df_2009 = pd.read_csv(files["2009"])
df_2015 = pd.read_csv(files["2015"])
df_2018 = pd.read_csv(files["2018"])

# ============================
# 2. 构造训练集 & 测试集
# ============================
df_train = pd.concat([df_2009, df_2018], ignore_index=True)
print(f"训练集样本数: {len(df_train)}")

# ============================
# 3. 取特征列 (第7列到倒数第二列)
# ============================
X_train = df_train.iloc[:, 7:].apply(safe_numeric, axis=0)
feature_cols = X_train.columns.tolist()
print(f"训练集特征数: {len(feature_cols)}")

# ============================
# 4. 计算两两相关性
# ============================
results = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        col_i, col_j = feature_cols[i], feature_cols[j]
        x, y = X_train[col_i], X_train[col_j]

        # 去掉 NaN/inf
        mask = (~x.isna()) & (~y.isna()) & np.isfinite(x) & np.isfinite(y)
        x_valid, y_valid = x[mask], y[mask]

        if len(x_valid) > 1:
            r, p = pearsonr(x_valid, y_valid)
            if abs(r) > threshold:  # 只保留高相关
                results.append({
                    "Var1": col_i,
                    "Var2": col_j,
                    "Correlation": r,
                    "p_value": significance_marker(p)
                })

results_df = pd.DataFrame(results)

# ============================
# 5. 统计每个变量“高相关次数”
# ============================
var_counts = pd.Series(dtype=int)
for _, row in results_df.iterrows():
    var_counts[row["Var1"]] = var_counts.get(row["Var1"], 0) + 1
    var_counts[row["Var2"]] = var_counts.get(row["Var2"], 0) + 1

var_counts = var_counts.sort_values(ascending=False).reset_index()
var_counts.columns = ["Variable", "HighCorr_Count"]

# ============================
# 6. 保存结果
# ============================
output_pairs = os.path.join(DATA_DIR, "HighCorrelationPairs_trainset.csv")
output_counts = os.path.join(DATA_DIR, "HighCorrelationCounts_trainset.csv")

results_df.sort_values(by="Correlation", ascending=False).to_csv(output_pairs, index=False, encoding="utf-8-sig")
var_counts.to_csv(output_counts, index=False, encoding="utf-8-sig")

print(f"\n✅ 高相关性变量对已保存: {output_pairs}")
print(f"✅ 变量高相关次数统计已保存: {output_counts}")


训练集样本数: 40669
训练集特征数: 107

✅ 高相关性变量对已保存: E:\Project-yqr\828update\BD\results\HighCorrelationPairs_trainset.csv
✅ 变量高相关次数统计已保存: E:\Project-yqr\828update\BD\results\HighCorrelationCounts_trainset.csv


## 计算贡献

In [28]:
import pandas as pd
import os
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor

# ============================
# 路径 & 参数
# ============================
DATA_DIR = r"E:\Project-yqr\828update\BD\results"
files = {
    "2009": os.path.join(DATA_DIR, "soc_filled_2009_average.csv"),
    "2015": os.path.join(DATA_DIR, "soc_filled_2015_average.csv"),
    "2018": os.path.join(DATA_DIR, "soc_filled_2018_average.csv"),
}
# --- 1. 合并训练和测试集 ---


# ============================
# 1. 读取数据
# ============================
df_2009 = pd.read_csv(files["2009"])
df_2015 = pd.read_csv(files["2015"])
df_2018 = pd.read_csv(files["2018"])

df_train = pd.concat([df_2009, df_2018], ignore_index=True)
df_test = df_2015.copy()

print(f"训练集样本数: {len(df_train)}")
print(f"测试集样本数: {len(df_test)}")

# --- 2. 排除 SOC > 120 ---
soc_threshold = 120
initial_train_count = len(df_train)
initial_test_count = len(df_test)

df_train = df_train[df_train['OC'] <= soc_threshold].copy()
df_test = df_test[df_test['OC'] <= soc_threshold].copy()

print(f"\n排除 SOC > {soc_threshold} 后:")
print(f"训练集样本数: {initial_train_count} -> {len(df_train)}")
print(f"测试集样本数: {initial_test_count} -> {len(df_test)}")

# --- 3. 特征和标签 ---
X_train_orig = df_train.iloc[:, 7:].copy()
y_train_full = df_train['SOC'].copy()

X_test_orig = df_test.iloc[:, 7:].copy()
y_test = df_test['SOC'].copy()

covariate_names = X_train_orig.columns.tolist()
print(f"\n原始特征数量: {len(covariate_names)}")

# --- 4. KNN 插补 ---
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train_orig), columns=covariate_names)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_orig), columns=covariate_names)
print("完成 KNN 插补")

# --- 5. 随机森林建模，计算特征重要性 ---
print("训练随机森林模型，计算特征重要性...")
initial_params = {
    'n_estimators': 100,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'random_state': 66,
    'n_jobs': -1
}

model_rf = RandomForestRegressor(**initial_params)
model_rf.fit(X_train_imputed, y_train_full)

feat_importances = pd.Series(model_rf.feature_importances_, index=covariate_names).sort_values(ascending=False)
print("\n特征重要性排序:")
print(feat_importances)

# --- 6. 保存结果到 CSV ---
feat_importances.to_csv("feature_importances.csv", header=["importance"])
print("\n已保存特征重要性到 'feature_importances.csv'")


训练集样本数: 40669
测试集样本数: 21859

排除 SOC > 120 后:
训练集样本数: 40669 -> 37690
测试集样本数: 21859 -> 20461

原始特征数量: 107
完成 KNN 插补
训练随机森林模型，计算特征重要性...

特征重要性排序:
N           0.294835
CEC         0.029098
CLC         0.027523
pH_H2O      0.019490
K           0.019213
              ...   
cwn_ecf     0.001297
csdi        0.001217
tx3tn3      0.001006
tnltm20     0.000936
txb3tnb3    0.000392
Length: 107, dtype: float64

已保存特征重要性到 'feature_importances.csv'


## 计算VIF

### 第一轮

In [43]:
import os
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ===============================
# 配置
# ===============================
BASE_DIR = r"E:\Project-yqr\828update\BD\results"
IMP_FILE = os.path.join(BASE_DIR, "selected.csv")
OUT_VIF = os.path.join(BASE_DIR, "selected_VIF_by_category.csv")

DATA_DIR = r"E:\Project-yqr\828update\BD\results"
FILES = {
    "2009": os.path.join(DATA_DIR, "soc_filled_2009_average.csv"),
    "2015": os.path.join(DATA_DIR, "soc_filled_2015_average.csv"),
    "2018": os.path.join(DATA_DIR, "soc_filled_2018_average.csv"),
}

CATEGORY_MAPPING = {
    "土壤性质": "土壤理化",
    "地形": "地形",
    "土壤水分": "土壤水分",
    "土地利用": "土地利用",
    "碳投入": "植被",
    "氮投入": "氮投入",
    "平均气温指标": "平均气候",
    "平均降水": "平均气候",
    "能源指标": "平均气候",
    "降水指标：日数": "极端降水",
    "降水指标": "极端降水",
    "高温指标：生长季": "极端高温",
    "高温指标：日数": "极端高温",
    "高温指标：日较差": "极端高温",
    "高温指标：热浪": "极端高温",
    "高温指标：基础气温": "极端高温",
    "高温指标：百分比": "极端高温",
    "高温指标": "极端高温",
    "干旱指标：日数": "极端干旱",
    "干旱指标：干旱浪": "极端干旱",
    "干旱指标": "极端干旱",
    "低温指标：日数": "极端低温",
    "低温指标：冷浪": "极端低温",
    "低温指标": "极端低温",
    "湿润指标：湿浪": "极端湿润",
}

# ===============================
# 辅助函数
# ===============================
def safe_read_csv(path):
    for enc in ["utf-8-sig", "utf-8", "gbk", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"无法读取文件: {path}")

def calculate_vif(df_vars):
    """计算 VIF，返回 float 列表"""
    if df_vars.shape[1] <= 1:
        return [np.nan]*df_vars.shape[1]
    X = df_vars.values.astype(float)
    vif_values = []
    for i in range(X.shape[1]):
        try:
            vif_val = variance_inflation_factor(X, i)
        except Exception:
            vif_val = np.nan
        vif_values.append(float(vif_val))
    return vif_values

# ===============================
# 主程序
# ===============================
def main():
    # 1. 读取变量列表
    df_imp = safe_read_csv(IMP_FILE)
    var_col = next((c for c in df_imp.columns if "variable" in c.lower()), df_imp.columns[0])
    cat_col = next((c for c in df_imp.columns if "category" in c.lower() or "分类" in c),
                   df_imp.columns[2] if df_imp.shape[1] > 2 else df_imp.columns[-1])
    df_imp = df_imp.rename(columns={var_col: "Variable", cat_col: "Category"})
    df_imp["MajorCategory"] = df_imp["Category"].map(CATEGORY_MAPPING).fillna(df_imp["Category"])

    # 2. 合并三年数据
    all_data = []
    for year, path in FILES.items():
        df_year = safe_read_csv(path)
        all_data.append(df_year)
    df_data = pd.concat(all_data, axis=0, ignore_index=True)

    # 3. 按分类计算 VIF
    vif_list = []
    for major in df_imp["MajorCategory"].unique():
        sub_vars = df_imp[df_imp["MajorCategory"] == major]["Variable"].tolist()
        sub_vars = [v for v in sub_vars if v in df_data.columns]

        if len(sub_vars) == 0:
            continue

        sub_df = df_data[sub_vars].apply(pd.to_numeric, errors="coerce").dropna(axis=0, how="any")
        vif_values = calculate_vif(sub_df)

        for var, vif_val in zip(sub_vars, vif_values):
            vif_list.append({
                "MajorCategory": major,
                "VariableName": var,
                "VIF": float(vif_val)
            })

    # 4. 输出
    vif_df = pd.DataFrame(vif_list)
    vif_df["VIF"] = vif_df["VIF"].astype(float)
    vif_df.to_csv(OUT_VIF, index=False, encoding="utf-8-sig")
    print(f"✅ VIF 计算完成，保存至 {OUT_VIF}")

if __name__ == "__main__":
    main()


✅ VIF 计算完成，保存至 E:\Project-yqr\828update\BD\results\selected_VIF_by_category.csv


### 第二轮

In [53]:
import os
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ===============================
# 配置
# ===============================
BASE_DIR = r"E:\Project-yqr\828update\BD\results"
IMP_FILE = os.path.join(BASE_DIR, "final_selected_variables.csv")  # 注意这里改了
OUT_VIF = os.path.join(BASE_DIR, "selected_VIF_second.csv")

DATA_DIR = BASE_DIR
FILES = {
    "2009": os.path.join(DATA_DIR, "soc_filled_2009_average.csv"),
    "2015": os.path.join(DATA_DIR, "soc_filled_2015_average.csv"),
    "2018": os.path.join(DATA_DIR, "soc_filled_2018_average.csv"),
}

CATEGORY_MAPPING = {
    "土壤性质": "土壤理化",
    "地形": "地形",
    "土壤水分": "土壤水分",
    "土地利用": "土地利用",
    "碳投入": "植被",
    "氮投入": "氮投入",
    "平均气温指标": "平均气候",
    "平均降水": "平均气候",
    "能源指标": "平均气候",
    "降水指标：日数": "极端湿润",
    "降水指标": "极端湿润",
    "高温指标：生长季": "极端高温",
    "高温指标：日数": "极端高温",
    "高温指标：日较差": "极端高温",
    "高温指标：热浪": "极端高温",
    "高温指标：基础气温": "极端高温",
    "高温指标：百分比": "极端高温",
    "高温指标": "极端高温",
    "干旱指标：日数": "极端干旱",
    "干旱指标：干旱浪": "极端干旱",
    "干旱指标": "极端干旱",
    "低温指标：日数": "极端低温",
    "低温指标：冷浪": "极端低温",
    "低温指标": "极端低温",
    "湿润指标：湿浪": "极端湿润",
}

# ===============================
# 辅助函数
# ===============================
def safe_read_csv(path):
    for enc in ["utf-8-sig", "utf-8", "gbk", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"无法读取文件: {path}")

def calculate_vif(df_vars):
    """计算 VIF，返回 float 列表"""
    if df_vars.shape[1] <= 1:
        return [np.nan] * df_vars.shape[1]
    X = df_vars.values.astype(float)
    vif_values = []
    for i in range(X.shape[1]):
        try:
            vif_val = variance_inflation_factor(X, i)
        except Exception:
            vif_val = np.nan
        vif_values.append(float(vif_val))
    return vif_values

# ===============================
# 主程序
# ===============================
def main():
    # 1. 读取变量列表
    df_imp = safe_read_csv(IMP_FILE)
    # 保留手动=否的变量
    if "手动" in df_imp.columns:
        df_imp = df_imp[df_imp["手动"] == "否"].copy()

    var_col = next((c for c in df_imp.columns if "variable" in c.lower()), df_imp.columns[0])
    cat_col = next((c for c in df_imp.columns if "category" in c.lower() or "分类" in c),
                   df_imp.columns[2] if df_imp.shape[1] > 2 else df_imp.columns[-1])
    df_imp = df_imp.rename(columns={var_col: "Variable", cat_col: "Category"})
    df_imp["MajorCategory"] = df_imp["Category"].map(CATEGORY_MAPPING).fillna(df_imp["Category"])

    # 2. 合并三年数据
    all_data = []
    for year, path in FILES.items():
        df_year = safe_read_csv(path)
        all_data.append(df_year)
    df_data = pd.concat(all_data, axis=0, ignore_index=True)

    # 3. 按分类计算 VIF
    vif_list = []
    for major in df_imp["MajorCategory"].unique():
        sub_vars = df_imp[df_imp["MajorCategory"] == major]["Variable"].tolist()
        sub_vars = [v for v in sub_vars if v in df_data.columns]

        if len(sub_vars) == 0:
            continue

        sub_df = df_data[sub_vars].apply(pd.to_numeric, errors="coerce").dropna(axis=0, how="any")
        vif_values = calculate_vif(sub_df)

        for var, vif_val in zip(sub_vars, vif_values):
            vif_list.append({
                "MajorCategory": major,
                "VariableName": var,
                "VIF": float(vif_val)
            })

    # 4. 输出
    vif_df = pd.DataFrame(vif_list)
    vif_df["VIF"] = vif_df["VIF"].astype(float)
    vif_df.to_csv(OUT_VIF, index=False, encoding="utf-8-sig")
    print(f"✅ VIF 计算完成，保存至 {OUT_VIF}")

if __name__ == "__main__":
    main()


✅ VIF 计算完成，保存至 E:\Project-yqr\828update\BD\results\selected_VIF_second.csv


## 输出结果

In [48]:
# select_vars_full_safe_mark_fixed_with_vif.py
import os
import pandas as pd

# ===============================
# 配置（按需修改）
# ===============================
BASE_DIR = r"E:\Project-yqr\828update\BD\results"
IMP_FILE = os.path.join(BASE_DIR, "selected.csv")
CORR_FILE = os.path.join(BASE_DIR, "HighCorrelationPairs_trainset.csv")
SOC_CORR_FILE = os.path.join(BASE_DIR, "SOC_correlations_sorted.csv")
VIF_FILE = os.path.join(BASE_DIR, "selected_VIF_by_category.csv")  # 新增

OUT_SELECTED = os.path.join(BASE_DIR, "final_selected_variables.csv")
OUT_LOG = os.path.join(BASE_DIR, "selection_log_detailed.csv")

# 规则项
ALWAYS_DROP = {"csdi5", "wsdi5"}
DROP_SUBCATEGORIES = {"能源指标"}
HEATCOLD_DROP_PATTERNS = ["hwn", "cwn", "tn90", "tx90"]
AVG_TEMP_ONLY = {"tmm"}             # 平均气温：只保留 tmm（受保护）
CORR_THRESHOLD = 0.85

# 额外保护的小类（整类保留），若你的分类名称不同可调整
PROTECT_SUBCATS = {"高温指标：基础气温"}

CATEGORY_MAPPING = {
    "土壤性质": "土壤理化",
    "地形": "地形",
    "土壤水分": "土壤水分",
    "土地利用": "土地利用",
    "碳投入": "植被",
    "氮投入": "氮投入",
    "平均气温指标": "平均气候",
    "平均降水": "平均气候",
    "能源指标": "平均气候",
    "降水指标：日数": "极端降水",
    "降水指标": "极端降水",
    "高温指标：生长季": "极端高温",
    "高温指标：日数": "极端高温",
    "高温指标：日较差": "极端高温",
    "高温指标：热浪": "极端高温",
    "高温指标：基础气温": "极端高温",
    "高温指标：百分比": "极端高温",
    "高温指标": "极端高温",
    "干旱指标：日数": "极端干旱",
    "干旱指标：干旱浪": "极端干旱",
    "干旱指标": "极端干旱",
    "低温指标：日数": "极端低温",
    "低温指标：冷浪": "极端低温",
    "低温指标": "极端低温",
    "湿润指标：湿浪": "极端湿润",
}

# ===============================
# 辅助函数（保留原逻辑）
# ===============================
def detect_column(df, candidates):
    if df is None or df.shape[1] == 0:
        return None
    for c in df.columns:
        low = str(c).lower()
        for cand in candidates:
            if cand.lower() in low:
                return c
    return None

def safe_read_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    for enc in ("utf-8-sig", "utf-8", "gbk", "latin1"):
        try:
            return pd.read_csv(path, dtype=str, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path, dtype=str, encoding="latin1", low_memory=False)

# ===============================
# 读取输入文件
# ===============================
def read_inputs(imp_file, corr_file, soc_corr_file, vif_file=None):
    df_imp = safe_read_csv(imp_file)
    df_corr = safe_read_csv(corr_file)
    df_soc = safe_read_csv(soc_corr_file) if os.path.exists(soc_corr_file) else pd.DataFrame()
    df_vif = safe_read_csv(vif_file) if vif_file and os.path.exists(vif_file) else pd.DataFrame()

    # 标准化 df_imp
    var_col = detect_column(df_imp, ["variable", "变量", "var", "name"]) or df_imp.columns[0]
    imp_col = detect_column(df_imp, ["importance", "importance.score", "重要", "imp"]) or (df_imp.columns[1] if len(df_imp.columns) > 1 else df_imp.columns[0])
    cat_col = detect_column(df_imp, ["category", "分类", "class", "小类"]) or (df_imp.columns[2] if len(df_imp.columns) > 2 else df_imp.columns[-1])
    df_imp = df_imp.rename(columns={var_col: "Variable", imp_col: "Importance", cat_col: "Category"})
    if "Importance" not in df_imp.columns:
        df_imp["Importance"] = 0.0
    df_imp["Importance"] = pd.to_numeric(df_imp["Importance"], errors="coerce").fillna(0.0)

    # 标准化 df_corr
    v1_col = detect_column(df_corr, ["var1", "v1", "column1", "var_a"]) or df_corr.columns[0]
    v2_col = detect_column(df_corr, ["var2", "v2", "column2", "var_b"]) or (df_corr.columns[1] if len(df_corr.columns) > 1 else df_corr.columns[0])
    corr_col = detect_column(df_corr, ["correlation", "corr", "value", "cor"]) or (df_corr.columns[2] if len(df_corr.columns) > 2 else df_corr.columns[-1])
    df_corr = df_corr.rename(columns={v1_col: "Var1", v2_col: "Var2", corr_col: "Correlation"})
    df_corr["Var1"] = df_corr["Var1"].astype(str)
    df_corr["Var2"] = df_corr["Var2"].astype(str)
    df_corr["Correlation"] = pd.to_numeric(df_corr["Correlation"], errors="coerce").fillna(0.0).abs()

    # 处理 df_soc
    if not df_soc.empty:
        soc_var_col = detect_column(df_soc, ["all_variable", "all_var", "variable", "2009_variable", "2015_variable", "2018_variable"]) or df_soc.columns[0]
        soc_corr_col = detect_column(df_soc, ["all_corr", "all_correlation", "allcorr", "2018_corr", "2015_corr", "2009_corr", "corr", "correlation", "value"])
        keep_cols = [soc_var_col]
        if soc_corr_col and soc_corr_col != soc_var_col:
            keep_cols.append(soc_corr_col)
        df_soc_small = df_soc[keep_cols].copy()
        df_soc_small = df_soc_small.rename(columns={soc_var_col: "All_Variable"})
        if soc_corr_col and soc_corr_col in df_soc_small.columns:
            df_soc_small = df_soc_small.rename(columns={soc_corr_col: "All_Corr"})
            df_soc_small["All_Corr"] = pd.to_numeric(df_soc_small["All_Corr"], errors="coerce").fillna(0.0).abs()
        else:
            df_soc_small["All_Corr"] = 0.0
        df_soc_small["All_Variable"] = df_soc_small["All_Variable"].astype(str)
        df_soc = df_soc_small[["All_Variable", "All_Corr"]].copy()
    else:
        df_soc = pd.DataFrame(columns=["All_Variable", "All_Corr"])

    # 处理 df_vif
    if not df_vif.empty:
        vif_var_col = detect_column(df_vif, ["variable", "变量", "var", "name"]) or df_vif.columns[0]
        vif_value_col = detect_column(df_vif, ["vif"]) or df_vif.columns[1]
        vif_cat_col = detect_column(df_vif, ["majorcategory", "category", "class"]) or (df_vif.columns[2] if len(df_vif.columns) > 2 else df_vif.columns[-1])
        df_vif = df_vif.rename(columns={vif_var_col: "Variable", vif_value_col: "VIF", vif_cat_col: "MajorCategory"})
        df_vif["VIF"] = pd.to_numeric(df_vif["VIF"], errors="coerce").fillna(0.0)
        # 新增 VIF_Level
        df_vif["VIF_Level"] = df_vif["VIF"].apply(lambda x: "高" if x > 20 else "")
    else:
        df_vif = pd.DataFrame(columns=["Variable", "VIF", "MajorCategory", "VIF_Level"])

    return df_imp, df_corr, df_soc, df_vif

# ===============================
# 后续函数保持原逻辑
# apply_rule_deletions / remove_high_corr_by_major / save_final
# ===============================

# 这里为了简洁，我直接复用你原脚本的函数
# 只在 save_final 中增加 VIF 合并逻辑
def save_final(df_imp, rule_remove, corr_remove, log_rule, log_corr, df_soc, df_vif):
    df_final = df_imp.copy()
    df_final["MajorCategory"] = df_final["Category"].map(CATEGORY_MAPPING).fillna(df_final["Category"])

    # 合并 SOC
    if not df_soc.empty:
        df_soc_safe = df_soc.copy()
        if "All_Variable" in df_soc_safe.columns:
            df_soc_safe = df_soc_safe.rename(columns={"All_Variable": "Variable"})
        for col in list(df_soc_safe.columns):
            if col != "Variable":
                df_soc_safe = df_soc_safe.rename(columns={col: f"SOC_{col}"})
        df_soc_safe = df_soc_safe.loc[:, ~df_soc_safe.columns.duplicated()]
        df_final = df_final.merge(df_soc_safe, on="Variable", how="left")

    # 合并 VIF
    if not df_vif.empty:
        df_final = df_final.merge(df_vif[["Variable", "VIF", "VIF_Level"]], on="Variable", how="left")

    # 合并原因字典
    rule_reason_map = {r["Variable"]: r["RuleReason"] for r in log_rule if r.get("RuleReason")}
    corr_reason_map = {r["Variable"]: r["CorrReason"] for r in log_corr if r.get("CorrReason")}

    df_final["ToDelete"] = df_final["Variable"].apply(lambda v: "是" if (v in rule_remove or v in corr_remove) else "否")
    df_final["RuleReason"] = df_final["Variable"].map(rule_reason_map).fillna("")
    df_final["CorrReason"] = df_final["Variable"].map(corr_reason_map).fillna("")

    # 保存
    df_final = df_final.sort_values(by="Importance", ascending=False)
    df_final.to_csv(OUT_SELECTED, index=False, encoding="utf-8-sig")
    print(f"\n已保存最终表（全量变量 + ToDelete 标记 + VIF信息）: {OUT_SELECTED} （共 {len(df_final)} 个）\n")

    # 保存日志
    df_logs_rule = pd.DataFrame(log_rule)
    df_logs_corr = pd.DataFrame(log_corr)
    if not df_logs_rule.empty and not df_logs_corr.empty:
        df_logs = pd.concat([df_logs_rule, df_logs_corr], sort=False, ignore_index=True)
    elif not df_logs_rule.empty:
        df_logs = df_logs_rule.copy()
    elif not df_logs_corr.empty:
        df_logs = df_logs_corr.copy()
    else:
        df_logs = pd.DataFrame(columns=["Variable", "RuleRemove", "RuleReason", "CorrRemove", "CorrReason"])

    df_logs["ToDelete"] = df_logs["Variable"].apply(lambda v: "是" if v in (rule_remove | corr_remove) else "否")
    df_logs.to_csv(OUT_LOG, index=False, encoding="utf-8-sig")
    print(f"详细日志已保存：{OUT_LOG}")

    total = len(df_final)
    del_count = (df_final["ToDelete"] == "是").sum()
    keep_count = total - del_count
    print("===== 最终统计 =====")
    print(f"总变量数: {total}")
    print(f"标记为删除 (ToDelete=是): {del_count}")
    print(f"标记为保留 (ToDelete=否): {keep_count}")
    maj_counts = df_final[df_final["ToDelete"] == "是"]["MajorCategory"].value_counts().to_dict()
    print("\n按大类统计（被标记删除数量）：")
    for k, v in maj_counts.items():
        print(f" - {k}: {v}")

    return df_final, df_logs

# ===============================
# 主程序
# ===============================
def main():
    print("开始读取输入文件...")
    df_imp, df_corr, df_soc, df_vif = read_inputs(IMP_FILE, CORR_FILE, SOC_CORR_FILE, VIF_FILE)
    print("读取完成。\n")

    rule_remove, log_rule = apply_rule_deletions(df_imp)
    corr_remove, log_corr = remove_high_corr_by_major(df_corr, df_imp, df_soc, rule_remove)
    df_final, df_logs = save_final(df_imp, rule_remove, corr_remove, log_rule, log_corr, df_soc, df_vif)

    print("\n脚本执行完成。")

if __name__ == "__main__":
    main()


开始读取输入文件...
读取完成。

规则 -> 标记删除: txm ；原因: 平均气温小类规则（非 tmm）
规则 -> 标记删除: cddcold18 ；原因: 小类删除: 能源指标
规则 -> 标记删除: gddgrow10 ；原因: 小类删除: 能源指标
规则 -> 标记删除: hwm_tn90 ；原因: 热浪/冷浪规则匹配 `tn90`
规则 -> 标记删除: hddheat18 ；原因: 小类删除: 能源指标
规则 -> 标记删除: hwa_tx90 ；原因: 热浪/冷浪规则匹配 `tx90`
规则 -> 标记删除: tnm ；原因: 平均气温小类规则（非 tmm）
规则 -> 标记删除: hwa_tn90 ；原因: 热浪/冷浪规则匹配 `tn90`
规则 -> 标记删除: hwf_tx90 ；原因: 热浪/冷浪规则匹配 `tx90`
规则 -> 标记删除: hwf_tn90 ；原因: 热浪/冷浪规则匹配 `tn90`
规则 -> 标记删除: wsdi5 ；原因: ALWAYS_DROP 无条件删除
规则 -> 标记删除: hwd_tn90 ；原因: 热浪/冷浪规则匹配 `tn90`
规则 -> 标记删除: hwd_tx90 ；原因: 热浪/冷浪规则匹配 `tx90`
规则 -> 标记删除: hwn_ehf ；原因: 热浪/冷浪规则匹配 `hwn`
规则 -> 标记删除: hwn_tn90 ；原因: 热浪/冷浪规则匹配 `hwn`
规则 -> 标记删除: csdi5 ；原因: ALWAYS_DROP 无条件删除
规则 -> 标记删除: hwn_tx90 ；原因: 热浪/冷浪规则匹配 `hwn`
规则 -> 标记删除: cwn_ecf ；原因: 热浪/冷浪规则匹配 `cwn`

规则检查完成：标记删除 18 个变量（ToDelete=是 的一部分）

相关性 -> 标记删除: tmlt5 ；原因: 与 tmge5(r=1.00), tnlt2(r=0.98), fd(r=0.98), tnltm2(r=0.95), tmlt10(r=0.91), id(r=0.88) 高相关 (r≥0.85); 最终保留 tmge5 ；(大类: 极端低温)
相关性 -> 标记删除: pH_CaCl2 ；原因: 与 pH_H2O(r=0.99) 高相关 (r≥0.85); 